# IRIS on Kaggle: real-data ML benchmark (M7) + current-state reruns (M4, M1, M6)

**What this does (press *Run All*, nothing to edit):** clones the repo, places the two attached OPEN datasets, ingests them in **provisional** mode,
then runs M4 (leakage checks), M1 (naive baseline), **M7 (next-day TDD: ridge / random forest / gradient boosting vs persistence, participant-held-out)** and M6.

**Before you run:** attach two private datasets with *Add Input*: HUPA-UCM (`HUPA0001P.csv ...`, the 25 *Preprocessed* files) and BrisT1D-Open (the 20 `P*.csv` files from `device_data/processed_state`).
Any folder layout works: files are found by name and checked against the SHA-256 hashes in the repo manifest. Settings: Accelerator **None**, Internet **On**.

**Honest scope.** Provisional run (source sign-off S15/S17 is still open). No menstrual-cycle labels exist in these data, so **nothing here says anything about the cycle**.
TDD under automated insulin delivery reflects controller behaviour as well as physiology. A model that does not beat a trailing mean is a legitimate result.

In [ ]:
import os, pathlib, sys, subprocess, shutil, json, platform, hashlib, csv
REPO_URL    = os.environ.get("IRIS_REPO_URL", "https://github.com/codes-and-games/Mensulin-IRIS.git")
BASE_COMMIT = "d625ea9ada8b439cccddbc9a1b17bd62932c382e"  # pinned repository revision for reproducibility
INPUT_ROOT  = pathlib.Path(os.environ.get("IRIS_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT   = pathlib.Path(os.environ.get("IRIS_WORK_ROOT", "/kaggle/working"))
VERIFY_HASHES = os.environ.get("IRIS_VERIFY_HASHES", "1") == "1"
def sh(cmd, check=True, tail=6000):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-tail:], (r.stderr or "")[-2500:])
    if check and r.returncode: raise SystemExit(f"command failed ({r.returncode}): {cmd}")
    return r

In [ ]:
os.chdir(WORK_ROOT)
sh(f"rm -rf iris && git clone {REPO_URL} iris")
os.chdir(WORK_ROOT / "iris")
sh(f"git checkout --detach {BASE_COMMIT}")
# ---- M7 files (embedded; written only if the repository does not already contain them) -----------------------------------
OVERLAY_B64 = """
UEsDBBQAAAAIAE9XSl1HdKcRYQcAAEASAAAdAAAAc3JjL2lyaXMvZmVhdHVyZXMvbmV4dF9kYXkucHmlV22P27gR/q5fQahIIx1s
NUlRBDDqAG52g1sg2RTJ5g7BwqBpidYSkUQdSXnXDZLf3mcoyZLWTg5FjWRhDmeG8/rMOAzDa/ng5pk4sJuLC2abWpq9sjJjTmwL
ySL5AIoqZeXYu5dxEgSXe2kObCeFa4xkesfcnWRG37OdNiwVhawyYRgpdExZluqybhz0vb9++5ntjC7pzjLrjEpdcWBbCUEJ5miz
sXdq56Ln8WbDdMWEFy6kkwFsmOO/BbV/YsasBkuprFVV7h+0jjT3FFFlZEAlYTDbGpXlMotnTD6ksnZss1GW30v5Bcrw3P2dSu+I
XbDaaDzlDvAtIN+cMLl09MBTi79O9ppxaVrjv1T6vmIKJmd7UaUyYdfa3ZERxMIaKxGIQ4p4FmIrCzvzKirN9qJoJOlSbZQWgQ+O
uxOOCUjC2KLJZDZjd2IvScLIVBtQKF0zhpD7C8rdP5fsGYt+/fTv1fzT63cLVjdlDZtsU8CwSjtErqacZTFijshKdi2u2Z2yTptD
EoRhGPj0cL5riI9zskobWFJBXDilKxsEHa2C9gMTiG/dk2o4BQL+1VkQ/H51ffH+d35x+Wb16e0NW7KXwbura/7r1ceb9x8+j+h/
D4K/LJilvM53RkrYWWXKB7qrMssiQZEX8z8aUSikRjSZchRwStCoREtUQYb8weTDEMVSW6q0vjAoT16dlS4OXq+uL64uVjeX/M3l
6ubTh8uPsCkKGD5hIfLn3MiCuywLZ/78Ynw2uihe8nQ/un/OU2G2xDQlwDqjpA1nI82lFBXPiybVVvIyzwYRpwyvUzect4/O4nge
yhi646Bzgq9+W129Xf3r6u3VzWc49PWXX77uFiyshXWhb9UdxeHU+W9TlRDxxc2F4zVqBz2LMgi/BYHPJGkO4TxviLOLgvc3JwJ9
O1L6CCwenXF/GggwdecuYv4T9nFZ+K8k2kdm4b96ihgowpCpQSYBU7AS7aZQOIcoQyzqLLlAEbwxopQxm79irgHY3IL8UZJhM0be
rhdtwsJw1UEA9Ro6trEeIHWFylItChIGUS9XfdN2rDtVqQ42iPKK+vQZQvd03J19Yz+d+WN9d7BKFzo/xAm1JhkBxZD6nu1uww4Y
+A7lEK4TZPVQy2irdRF7VngLVvjiNEerwqE0Irk2V2vAoDHa2GWYamlSGbZSnaFLNHWibHuKINLe1triijTDA09JdVM5In495ihs
YZoTjnEH2KBcqspFQG1YEMdDOnsnMr49cDSmtK71pxWIvsPfOLFNGU2k+vBS0/Gup3sRitBf2ffW8j+T/Y80mmvDK5kD3PaSOwO8
waWwHCngfUamyrsY4RXE48wTvsa8fgrBD2RPRL/5v0YC7ip2yjzrQt3V8rZRRQa7Hxw9wv2oPqnpGbtXVabvvQXI0RSQZ4DDinfo
37OcAehpZ4yUT5vjQ2t5JAuVK+oLbARHq+OEXfb0JfMB6qeqn2IthVpDt4gOpNkr3djJPtEOzVdLjO6R6ZuNN2FQ4QeoKDWAfqJq
s2nDgWE/1moT9loXTVnZRT8TZqyrYYUWpjnECw2RGTv4tDYz1mKw/+5f93Ohu2tngkc0T2GRNwMaYbDEcsD+xpwRCn2fz4mNbfEo
TtLG3kNiP8XlIwhQX2e7JNX1IWr7MrsNOZkZro8tT0eHkRjhbvAgXLcCgKssyZBTlVK9R7fh0WEC0E5ZnGCIRvHiWNow2kr2G60s
l4QeUXjUwaKzIYt9GfRzumXxu6av2A50fOpmA5ZMgTqeIN+fA1+tZCpJze26hTSMupr0+7mfJbnRTb09RBOXLdaX5Y1p5MjbHDry
hG6439Js1AUmPrLAwzbi3mMjAGFRfkxGgiqNsHCOKeKBKNhy/liGFyNNZavn2F7Ai0w+LAfRgdWv2RrVWIGPfPIdmwAqSjvOlg/t
yQzIbyH7I/jvP/pL36YJ0nibJ96aNdUNpNlyybo5wrDLkg3phG+iq7wlO6ky98k9bcOR/hJ3Fh0ruA9BmRjpdURUPMcrWlro1o8v
hLH/mTBUJnqOXAVfQt/RWlHb7S3M0X6oM7sc4UacUO+N3jfWZf+zDsiMVKSDoX7zOWNp2pma/l+26vPlMgma9p1PeOZxIR8dE0Xp
erYe8w7t4FFEZY8UHSGEjBcFPDc7jzDhk8/zJ+X8SRZO3u6w0vP3iRvfDxDaPohwPLp+cUZDF9AXk6ceI66XIOLJe/3q3r8IKD7D
92LC99iYU5HJrwD/NpXSWdWjXwfEGKWdFb4s4q5B2hrBjhWfl+4X5/VQagPttOR62dMlu1PQL9o/ke337j4X2L1/xr2dcm9/zi2m
3OJH3KOfJW3wUIYYJnpHVFoM/hH382BXaOFGAOKHQiLqGsKR7pbdpujgO9UVhljUcs2Yyiv8qOdtT/mp4PlptQE/iY3KO8GmWIko
poXN34wL++TytFR7ltE+7WOA5ajdoT0Oe4dpkfzB690uOVHSr2LDnkjD+KiJrsdi3e5JD3g0p/s18BiI0YYioh/WbTyO6+h/AVBL
AwQUAAAACABPV0pd26nLT1kCAABoBAAAJgAAAHNyYy9pcmlzL2V2YWx1YXRlL2NsdXN0ZXJfYm9vdHN0cmFwLnB5fVTBitswEL37
KwYfigyO62zpJZBCsyyl0NIcelsWI8cTW0TWeCV5s2npv3ckmSzbluZizejNm5nn5+R5fqtn59FiB6K9wITWkSmgJfLOWzltwKKT
46QR9nff9l/uSjD4hBY6eXEgOjrMIxoP67p6V1R5nmdHSyM0zXH2s8WmATVOZD1IY8hLr8i4LFtyZh6nC0gHZsqyrMMjHNI0zbV/
405Ka9FbmidXgmxdg9Y2I3WoX8JWOizBmn7DVJWVpqOx+oQGrfRkS9htQPGQW7ip65rL9DTIDRw1yZCsq/p9AasP0KmD32TAP15k
T6SxW339eAdxBgauYQUcp+7wNp5Da60Mwln5AWRQ8MCCKBbs9jNEMRYFlekXfV0CW5y0PGDQr4pd9+SUV094begHhNRsYJU0nVn4
Kc4VVic9ewTenywjpYnw6zyUYm6NYOns4ssJXXomZpWkk9bKyyJtEa9wLAHb1/d/SB5FK8p/IdJbSIBIR6dEpdxRGeVR4FjAm9eZ
NkH78qV5f0+nhxAvzzY8I0p17AFlnhLtbNTjjIJLLbLZTMM3rC9uv9sZE61jTrcs1CpzoNl4wbASzqj6wbstj1T+53YZLyyi0Qge
ICUmSoaK24p1VbM13Fi5eRQFO8O16VgsYz8zlO1ZcRH2PKNgF554NPUDt2LH5wUZfB+MtvDdc+VDZJLPym3XifqvbCzVVMKg0qqP
s4wWFIGvhPtoeK69KaOHr+FDKk3ywc88+i7fpO1KyA+q0ZQvX4rQVKTcoK65QYWcaSYk/o/I43cmTiG3W4Jd8Sv7DVBLAwQUAAAA
CABxV0pd3H/eWfkBAABuAwAAKgAAAGV4cGVyaW1lbnRzL003X25leHRfZGF5X3RkZF9tbC9jb25maWcueWFtbG1S72vbMBD97r/i
aAm0H9o52bIyjxWyH4zC0gS6fRZydE5EZcnTyUn83+/OUbaVTWCDT+/e3XvPeOww2hZ9qmB5pzwekzJ6UMkY1bqCEE0Fs3L2dlpO
y6JBnfqIVBUAB+tNOAiYKrjjQmu92llKIQ65+lqq+qhym2otkfXbCsrbOYznEoxO+uZnr51NA8TeITT2iAZqbEJE0F6qvgINmQUy
C1gP9/NyAqEBdHZra+6N4UBSkDZhJkxgCUwMXcekV9obcGG7RXNdyL6CV+yAPBR4DIvc7EWeV9TxTqxizl8de6TjUME31Htcefwa
Q9+t+pRV4B7jACcSGdh7ds7Liryyfw9ph8AQYzcJekLiAqNos8MWIXg3FG0w6Og0GSnxODaSxPuSa9GaLSrtup2WHW/Log4hUYq6
k5aPI64cJdGzdU4sLmfw8lzC4unpx3L9/WH1+B+XUbtsdQKHmhLMJmzWASN0IThGLxdfeHHtRzk1psRXZNuOja/Zamc9Flmlkigr
uGidqjk34jcldQKrMxhs04xc7+aTbN7NxvXEvLLbWSF8epBMZfooDvb09wZ8tZ6+Wt+Bs2ws/8k2ohtA12GPUMLi8fOI7oL1KRPc
f4DfVsFVDjfHcX1RUBueUXz9E8q/sXDxRQbnFAoaPM9LdnMK86RLWt4UvwBQSwMEFAAAAAgAcVdKXf5X20IeDgAAdScAACUAAABl
eHBlcmltZW50cy9NN19uZXh0X2RheV90ZGRfbWwvcnVuLnB5tVppb9tGGv6uXzHgAhsyldg4CzSAUhVwYqdrrO0YidtNIQjEiBxK
A/Mqh5StGt7fvs87M7x0ON0jBhJRw5n3vkeO41y9mbJYlqpiqi5EuZFKROzqki1FFq5TXt5NWVGKSIYV4wwbVJ69UOz6/Mvt5Oz0
N5YvFQ7hSJVXPGERl8mWRbkSzL09O/NYXOYpK7iqJnmGN7HgVV0K5Y9uebkSFZsxbHMjvmWVx0qR8EpuBICxai06dFUJuDJbsTcT
2poKntE5RgcV+xFn3zZUyjxTjJeChXm2EWUF0pY8vCOQdSYr9T0B4FnEVJjjxIivuMzAPeGSqgLXIP3mZMq2At9K7PbGmpiWBo3d
vXnjaTCcZVyTjNdZ9/rqlQd8kSjBfFnJUBY8qyZrkUSTvK7Y+19H7qXgG/ExEz+XeV18rEkWRSkh8u2Y6bV/fMiTCKtKgJcI657P
brRIJmFSE3XEXJ5XCsgL9v5CjUHOvazWMpsY2TG1ruM4EdGkMvIuEh6KZW5IH8XyASAgJq3fsk7AO3DFcuWxOyEKzXiYcJmydZ5B
ID67zlm4DbEx4UuRGFHXsJkp9krFFCnk+uPt3y+uf2Z8SbwSjFRAxmUNC9GHxxo/aVDLaMTrKk856Qq6qCFlFokEUi23sAmQH1aK
9FmVeZJApEux5huZ1yXjit2LJKHPYr1VMk/y1dZ3HGck0yIvK5bVabGl11nRLEETERboSDTS9qnuEsHLzMeGuhLM7vuMz0Rc6LVy
uBEU4jNIc5DZbP8ko5UYbitkIWhrsyXldyJoFne2lqIo81AoRTbUUFARqWX0OeRge2ROyFIqP8zTNM988RCKwph8cySUIqtkLMN3
SR7eNYT3DwHPRmScDN0eOt/IiAz/fcKV6h2AwmGPVV76S6400S2eazL6K5j6we1pErTxoz2yKnlExAUwWezMVmNWgr88DWJ4oqr6
kDY8qWEPvrXzoLNyC2zvRaDuZJIchLHpKIBXiSgIN4GNFuN2TRUJokPvfBOqmsPQ0x1fiWBVQyNkPcnqwG4/Ew9VQDHGHnt/en12
cXZ6ex58OD+9/eXT+ecxs0/B6a+nF5en7y4uL25/G7NlLZMoaM4HFV8mooehyvOkJUY8wL9lStJM5JKoeUhGo88XVzeX54gYrnP1
KtBxKaBw5IyZc3MS9EKcXnkTNDHN7PJGiPt0+NNJUJI1ByXCcU57P70OBrrqvflbsKdY+9YbjUaRiBmgV6UMlYvAVnhs8hMj0U9H
DH8CCLdswgr9rRQQYsYencyZIhRUbiIyd+shAjtlqkRQYzlOcl65WeGr30v9SbS7gr18yV57emvKd3bqHfjkS+UK2qORNX/YX8At
w6o9cvLqFXvJ9g6y75mhZSm5OoBB7EEuX/e2nYBNIrtOG2oB0C64JIOtAeN5lpenRn42aDdG68Y8hJdBml/IhSBqa8aI/5k1ZLzI
Vp6RMcLhLWUnhnxg0oBq8wL75wXFaiZ4uO5ymE0ebiXI3BC1YSR3iMnwtHwrIsqIeNbwdGasBWtTWyFyhE0dgwn5Fw4Nf4H1BjoU
u1oWNl3jDfiP6yTRitasADbWEJ48fRzmBviED/aw46zuLtOWX/rbViWga4jzqlwg9gE30K70ujlIL9oDhKiQlIAIP0qF32vhYncP
Jv3JB0P0/RrZ19XQZnTOm79avCWsc/mwIMzZyocU07oiEmDE5o3XAiP+55WgvVadrufHsnK/cKJrTLA8v9E4LQp72joJvbIGUtaZ
G1YPltQwXgEovvt4Mif4PTwPYR/rD4kPDUSBUTFFGjo61mVDoJNzYDL77ANPlDAoEYvGSMB1BsuZHQxUrsZBLgukc6cJh85i7txL
BI572oqv3pFNqcwC1A8kh2anQU2ZBzgTni4jzpSMQOqaV1NiBKUDQhx4AWuGC3qaT08gvkFO899/vLr55fb8bNxBgB8fidzf98Kr
8q/eOJ0p3o3ZhkzEyMKXcBDl9kxExuzOV3CySpFAXddBik7qSEQUK2Vmnz1TO26GtkUqQ/3i2yNu7DxunqwvUt2rpuzx7onCKu3+
C5vgD5SsKSHAre6Zi70owSuIDNX05Bv+aQqyAFqDblqFkhLL/F6RcVkDc6zREmUzsiNf+95y6zrWAiXE4Sv5h3DNTpVyFHQzOjGn
Uz8aPAsfZiQeRlbKFC/0zp7wD8iv29YIUmlLB9RHDfaJoa5ZSVgwI/3r9oJYgJelOfU1OgG3NgAGDBvzf9F/PR5AH0o3i2zhw6BE
FWiS3ajMi9ktgqS1I1LVjM21Y1ttzZzTy0uYCExnJWawsmy28Z6xuEVrkRHiX0UbBrK1YJ2edDTa73bx0nGD1EnfBCZ2O4SelFoN
GcxsVNRZ9hkoJL8WBmmgwglDMalIoe2xQaOI/DOA+FDyVFBmuKfsaiw6oK+AMqcA4BLY4bpxJlT8jWhMqlj39KhxOJ61QBjNnspa
jmAPP3SiQhpE0NkrpXtUEAGxuBeEEe3eD/0uz1pYrQg9tS3o32pqazVR1PTYQGe4WFEmTVFV9T2dgRfwCfgHfMrA5zBRtLAIJ0jM
oSRJ7IQGG9kYryNZse+aCpaZCvabhoZUKrXr761NzhPw7+4XxnAbXhTJ1rXBHj1lBK/KODKjrYz8Ww1e96Yw5VgrPSbz3wdHGic6
EBPCebzwU/5Aip7Zcmw/BfGHwH4P6BxUg+jVeZpGAwYOUY7qjd4QXQcC0j06PESjRh8UD1DFMNdiYT+xR6LmRUPNi8X8xQFqXiye
NAl5Ktooj5QQP9ngZPQ8M0z3A5CHbxm5WJgndZqp2aOjX6A0bYhynjwfiVKuMjBRVLOeClopNTFOc+kd8mdNwbgFGujvAyfefeM0
MohLroc3OlO0losk2ZqN9eNk5YdrEd51oDZoYvgSjUy11bQda7C6CVGARhWxiqoXnaxBiCmMA6mCKopstO47E4pfCOebJtVjzrS1
KcfZEm3oKBb75TRNRppt9PxGt3TP7KeyVPde3+uzGtGqgdCPke1RnPpiN5CUUe5S8jcdmsncQqC2ykS/JKC1YdWnhyba4bKgnRio
pk5Af9tWe9PhCMalPr8Sqy2SDBSpG1btmzzEkcfdrnXaQhmMXVyAdEHPcLZCK3qA4/KkWPNZL0R09BroegOVp8Nm71iL/BUyBkdc
K8GeXGbZblf5XMf9FWR7x44hfDL6hJ+lguL43HW6iaTTU2a4MYo0/RdJZcxcZ3eyiRPXOeCaWGqHnNSjdCDsorOw6SMz8xw2iNbt
cmMt7/qm1o6CCN47bHlLXZhthagfW4lMlMSk66Cgoz6+PWF9Xc+EdbodN1NR+83SF9i2dY4Og/4RW22CUBRjIU2lo7QRXpcMbrSR
Ducy0wMjKbdRYTtic01P/Icoc+W62ZidkJjR/q8I2c7MYW/SMzUunfDVybFwcGgYNNVh4WnQHxv+dB6E0+13P5rNOe1aNE32kLV4
MLLQ1LOXXfzRSYzCx+PdlNmxCw1GdkrhmwZzR92SiijKfBlarj8z6hqjhtg2WQ7YuJjfLZoq4wDTxQDxkOVS1/OaQaP1mbUEHTpm
5vnly+EYzBuAIERL2kho/hwHQxq0EMjWD89FXQgbLGrV6KelfSxpDPvO24MFnvy6QO4V7mPsaBjBRgWP+tx8+nrxFKScjAumZV0S
lnF4Y5KbfaGkx+P71rLdh8fF05CqzjepShRZ5OJ5uAUFXyPDq8v/UT4wqGfE0w8ODTlfMYBmhD4j0GiXtAQgw1lfglpEs76stDBm
nVTGe7Qc/ctsP0enm2cCaWPZzNVU6gGWWYFX6Mk9r1RAVAZK59+gIz3PE7ejDoXrK12jdSywn2Zd9PZ6Vg7d7KIbamgnwt58tV/s
DIICmP6mBrVmu+QcuYMzNmX7tSNXam3d2dySgfw+FX1LOFgT2w2gwj4NaOzWHH0ZN9GXcGGegmRJvV5zP0md7VJUoJAZrbQGpSns
l6rLEo1elN8jEdkJbZNyjXV+myJWU3DTXWJqRerFSDVJtNAPbf7sDTFMY0PXta5OV03Vv/DboUNnLindUgy3kVlFyutVq/9hELdE
Dpy5P97ou3LrWM2Aozcxnqe6NBzEe6wBuXnT8rw/bN7hcGVHy93IuNgnsa3UZwUNNhuKqfGmtjeFXXWilLT0atGMZlJ99WCIRZCn
axOaLx65OCEuKBcbNo4k5MNt4cBhrKA7nw2WNPI1NO677/BtUydOrCVbB+7NPVt/fY4GK8khDXZgeZiE9uUuBVen5xp9L8S0JNhQ
RpbamSqBaNsrox6BaIo2hL61LV5zZbE3ej5gv/ouIXboQ2uQqq/NQSGUVO47vZ9KDNgdrjsIkpM8nsT0K4T+ryvcYTjx9gKQva+a
drdMzbVTihjLspyuwrOa7j2oyzeV/v8nAOm0tfyfSsFe4XmDotAbVIWZet3vOfb6H7rYMuPdLrp1dd1/d43aUzQ1x4euA5vKZae6
fj0+3vxYKDSoxVlvrxD/SrXUk1JReOOh1IwWqO3fLaBIOoMA1sVV9h1zAjuGaQwnML9m6dzP1ldxrx99BJ9Pzk6BFSy/SYnVip4r
JaK9eujH2X5BRGv+q9fenwiNJByvU01gf/Iy9NFWbc7zv/RpDvfm3+R4HOor586QEZBJhlqSlWoijswwnQ+nF5fnZ8H7j9e3nz5e
To/+vEit83u4PRTNS5i0UcVbFuWaBriPKGG9pqppSiBD5r2+UrJr8+azmw+wv7YvnWeLVYRQO7zRDa+eOXcqIJZkFuduDHdoa7sm
ss0e7cOTN6Ufw5kZECNkNKs0tRj9ZG5YibHlVr8zP6uiSm7KHokhyPZFlmfiBY1q/w1QSwMEFAAAAAgAcVdKXbwI8gr0BAAA0ggA
ACgAAABleHBlcmltZW50cy9NN19uZXh0X2RheV90ZGRfbWwvUkVBRE1FLm1kZVbvb9s2EP2ev+LQoZjTWfnRostWrx3cOGmDxrHh
OcP2yabEs82FIgWSsqMhf/weKdlNsA9RIJq8e/fu3aN+oPHFwvBjWEjRLIKUi1IfHT3RtWIt6YkurQlsAj1hLcsy6p54m9ausp4p
7nU+kK8rdlvlWdL4lnI2xaYU7uEDFcJQJXzIrNENKbOyrhRBWaw6lqoIJAhHvTU/erq7+muejYZ/k809wiHYfDRCtBDYUdggVHBq
q4SmXHjWyrCnGEm4oApVCRM8Gd5is2c2yIb9Qhll1r8n1F+byoYNe+UB3JwK6h2QDshYKppCM9mYiwFTU6GFKo/T2RtT1cEnUsoc
mWUHOwN1FESOg72v99Nhdn85pp/os1N+fj7KJhWb434XeRV5RZmTOdWg6iQF/sPWrmA6Jd4qCTgcs/qUaTKe3s+vRmnbXLg1oxWR
kl44xv6SQUgkyK4i4EjoVtna0wVY12ykcARwnnqGGWk/faR3tBVaybR8PCBfWDQBZBYPkazaqOBP8VsL7JpFqB1HJNN9Bz9Qwx7d
wCb0SxiZMkcKckZrGYsRkWONJm+Zgk0bYhvQrnXC3P/+GvduhVMix3to+i+iF8LldtNIJwK4fRHyeTiwC406xb7fUrLWdQFt9ml+
Mzudf8bfcDagSoOZpfKLHfMDyFnuaQstsRJZ2rrHVrKOVY/PyIiYskU9PU8tV0AYu9Q7YAWC6cV3UBdJEu2Z2Tk5JdcAM3tLDnzZ
kiJPPmDlHa2dkCpOWG6tD/Fwb6Ue0ZJNg1TQtSgZOVAZ1BnqqOTjFmQcTWd125yD/DOp/D9WIeDln9S7ZbHlieEvztbVpA5QiILW
QXNa+XZtMeWeCxul0kAPmsWDWDOta+HkYC/wAswBRBSKtcGjzmpAOxU2ymTtFvKberXSLLOOzEqLgnPb6ShVhKlM4+5qHW1jaKJR
lJFqepVDaf5V6kY37OjN9PwUrLa20Xbq1/ev6QAmO4Chy5u4fzy8Iv+gtCaMNzhVUExDn+iMhnej7heMwLJUZpHeltR7+7pPkjFv
qbqkYIi6gX4x/K42HdkjEcQBeFJ7LD+p97ePSICEAUcCgghPb5TxNZSAnqFc0OskyzfP3KFiWyHW2dn5+fQUz/fp+cuUNugXCa2z
f9lZ2ocprK5L45OdgQUndjASzR6ya80OOWGim8Yrq+1aYfgpBhhge/To5AGokPgR7MkIEnNb2NpEwIi6LOzGurBYabtbntAMCWSs
OJ7BJlsXm71ZXT1WXMRjtg6dHz473KdV6xkLUUsFiSeDgXq75u8XFnmziBk8v1hqxdTf3wy4JHC0k9KiaAXfeTEkgF2hvUliU1ol
4Y7AJFt4VuQ+ygrMvHCL2CsBoa9VwCyE6MlG4scTujcyclkHW4qWmJZ+xIUHuKYzNsi8QN0dHI0jOaNvMN7UiB1DZvh/aEczSDhi
/ufuoVqIB8+HgKPw8u8tLLnAjad82U0RigCzMTGojdyg7o4czJ/d+U7jvevhze3VaHE5uZvPJrd72SAsJnzFu/1V+jPuoHRtvbg+
e59vJ5ffrkbY6+wu0xCYJl/BnUNi6enoaFabD7SsGqjLUFaSwlV3EizM6AQjs2BIBD4TXe3/Xxf4gIidQostvhZQhYhjKKSkZZb5
0j7wMvpj1DNEn0fWID5cT9FkGoNCALVVZxrBZRcvwFGXxydH/wFQSwMEFAAAAAgAcVdKXcnONeVNAQAA3AEAADIAAABleHBlcmlt
ZW50cy9NN19uZXh0X2RheV90ZGRfbWwvZXhwZWN0ZWRfb3V0cHV0cy5tZF2QT0sDQQzF7/spAh5UbLdHwX8gdpXCtiu1Kp5mpzOp
Hd3OjEmmtt/eaRFRL4GE5L3fywFUm4hG0EJIEpPwGYxPlceNKKu3SqxVq64onsmJoAcJ0BJy6oQHlDwPLnJVzl4NRM875EELR1HT
R0KBE2jLSGGNXnuD5RsH3wI7i0YTH58VRR9MWAYStejCZw8WqCURKp2skx6wCdmpB2sk68zPQM23mUw0459RRMr6PYiEu20XfD6N
nTY4D8oELxRyjCqLbWGPCpmCHDLgOiNlQDCdZoabZnz/OKuGoL0FWSLkgIcMv4LAJPh+7m3a++wWGDQhsOhVzJ+8nzZPo4dRM7mu
IRDMqodZv5nUL2UxRW2h/U60exW5lc5El1CjXmPj8Y5Cik2S471/+y9BCwtHLOdglmjeoe3CaymbrLTIPrfXo7oaqptmMps2dVl8
AVBLAwQUAAAACAB3V0pdlK0x7WUGAADtEQAAJwAAAHRlc3RzL2xlYWthZ2UvdGVzdF9uZXh0X2RheV9mZWF0dXJlcy5wecVXa28b
Nxb9rl/BTpBitCvPSg4Md4OqQLBJimKbFmjdDwtBIKjhlTQwh5ySHMfaov+953Jm9LDlJM0WWMEY83F5H4f3xSzL3l2LNanYehJR
rQy9FI0K8cJZsxOlsyH6toyVsxPxX/Lu4ub1a7FVVpvKbiaiVIasVl5sVBMwNW2I5EmLlXMRR1UzESAWSoTa3ZLwrS2yLBtVdeN8
FLatm51QQdhmWGpAjgX8NXo0WntXi8pXoaA7ZVoVqehlyL0E0Z98tCHDbWXMEY/ezlBYuo9Sq91w9F+vfnj93etXN2/k2zevbn75
6c3PE7FqK6PlQCkTNke8onMmFGFn45ZiVQ6c9guyIR+c5bOj0Q+/vBNzscii1rLNJiJbqaBMP3TQuxuWyq/kZj8iG31FIc03tSzd
HXm1IZ7XpKzcmLZ0gWS90YYXY+VlU8Y0XB2GqhsuR6ORprWQjc5tr12Yv5iIpCNG04kIRHo+G78cCfw8ASx71qLcNoXHRbm6AEvV
mii93eR8fMwMB+770V5K928i3ldxK8tdaUgatSIT5m+VCTTulYwUohyuS2pq4GWSXRIfScqbChfNnPJeWb0GwGzaOE2BL02ExNq5
a8z1uiOLU6YAbVEZVy5m02VarlW4xQbIFllvdKWzpZjPceJkaSy+FHmj4Q3gHoFSTekUTyRYKsM038zhzMUNNkNUdZMzkxOKThvw
jRCr10Xpml1+WFzAfyD9MC4QobuG8rVxKnZ0wB8U5+9l2pGsnRdwVCvAokNtEFCw9Ww1QpgF4VDR2goH6nw2EVdT+AZvF6Gt815b
S+8/gDBz7ehUD/EiT0DvwXsKzUR1QGcgO8FruTBViPnjsB2LvyPMjNrM5D7WMLs8zLwz5lqm+OnWlsvu8vu7XzGK9H6R4/NxZZno
/6crOx5cCpkYDhGAuMQHGUPSr60yuTp1E6S004X+ftLJ0yhYZLtBovjiL8fjwHywBmo8E8gyqEB+Q1FUMZBZCw1BJYrNho7TAssE
omRLQk40bQ35QTae7iqHTMo+iBCQqAAVF6kOP6aA8SmHPMoZcPHLPkbgvZ+QN0IXpoGirKym+/zYxPGQ57tL8qDF6c7Uq/7ioCwn
mJOs4B+mjYsha2gyUeUpf87G4wLFbZ0yTfb8PxfP64vnOju5SiSBKpQGpYF5HjnYEhl+wbKXfW4BItfQIywWn6HK7ROqpDxzy3nG
89Uxul+NRbX+HHOflAHuoUjYL5dP2/44gCad0QWvcSY78itubxIZfMU6ZE4qndek06TZ7kLljNt0zsXtjlSe0t7KV3pD+pxbvejd
Cs7C149/SefF7HLJpbkPsbmYFlNx9vcMloYWfiwgCe1WwxVRI1ZUFNxa7AWCtfauyfciLqfLcYHqufdR3p7f+JbGR9yVqKsQECaH
Vm5gm0KhdK2N4WPx0IPfES+yyqIXY+TYwASr8zi6UbG6IwQm6jo21SnMXX2ddRaBgIVC+RySHnjKscge2Nkl4DwmSnABtY7Tp/+e
pS5XpN4QbSmh5QJGXVrqVYMHrLnPTIQPg/gpfTjZf140PxD4mXYdW1gUhUWIcr5lOxNzpFxkbyfQZSPpcs44UaK7heNUIr4R03Gh
jMnHqcPvKHZnth/AhtB5ErV/fhJql38KNZb3P4MG1A5QHWKG17b8fHEJsuNsQvd9DLDKMjkSGlrMt6hdzn+wCPVwXAGNno1cg3/K
FBzAf6JS2fv4FNpXf52Lnmc4wH4+ipPXsHofpDqG9PEDT1OkMgbccWVTXrYte9zLj3bELzr9BwK8L1TkR43qShbS9uE2uDN63BPj
4YR8mW/6LmpFkV1j/tQrNMdjOTH6G3L9V5P+geL5DY3uuu8qVE0f53D+7NDFJTUWWVkBwS4Ki9lV9wRfhZwlLLLEjm/gazGji9ll
2u62hnNfoyTxZ7+6rfYPyHQX9bVML3qJFz3qi+wWnaY81g1arbjtb+Hhi5np6R49HPzJxuHZfLqaDpbxnlE/2cizd9dHno5UU/PL
l+XOM1YBk6RWqnMTPh2kB47zvVZd7Ij0IkMh0lj1v7bwPcgrdOXFP/jJjBAKGQ8RtroqY9FTnfp9clngWW6pJgYUBey37Fvv2ubf
b53R3MN/T+qOfrSUVn9sY/Z7ghvHGtik/A5dsLL71175yZo1RpW0cmiBbUSfc15DdtGmTIpdjv4AUEsBAhQDFAAAAAgAT1dKXUd0
pxFhBwAAQBIAAB0AAAAAAAAAAAAAAKSBAAAAAHNyYy9pcmlzL2ZlYXR1cmVzL25leHRfZGF5LnB5UEsBAhQDFAAAAAgAT1dKXdup
y09ZAgAAaAQAACYAAAAAAAAAAAAAAKSBnAcAAHNyYy9pcmlzL2V2YWx1YXRlL2NsdXN0ZXJfYm9vdHN0cmFwLnB5UEsBAhQDFAAA
AAgAcVdKXdx/3ln5AQAAbgMAACoAAAAAAAAAAAAAAKSBOQoAAGV4cGVyaW1lbnRzL003X25leHRfZGF5X3RkZF9tbC9jb25maWcu
eWFtbFBLAQIUAxQAAAAIAHFXSl3+V9tCHg4AAHUnAAAlAAAAAAAAAAAAAACkgXoMAABleHBlcmltZW50cy9NN19uZXh0X2RheV90
ZGRfbWwvcnVuLnB5UEsBAhQDFAAAAAgAcVdKXbwI8gr0BAAA0ggAACgAAAAAAAAAAAAAAKSB2xoAAGV4cGVyaW1lbnRzL003X25l
eHRfZGF5X3RkZF9tbC9SRUFETUUubWRQSwECFAMUAAAACABxV0pdyc415U0BAADcAQAAMgAAAAAAAAAAAAAApIEVIAAAZXhwZXJp
bWVudHMvTTdfbmV4dF9kYXlfdGRkX21sL2V4cGVjdGVkX291dHB1dHMubWRQSwECFAMUAAAACAB3V0pdlK0x7WUGAADtEQAAJwAA
AAAAAAAAAAAApIGyIQAAdGVzdHMvbGVha2FnZS90ZXN0X25leHRfZGF5X2ZlYXR1cmVzLnB5UEsFBgAAAAAHAAcAVQIAAFwoAAAA
AA==
"""
if not pathlib.Path("experiments/M7_next_day_tdd_ml/run.py").exists():
    import base64, io, zipfile
    zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(OVERLAY_B64.split())))).extractall(".")
    print("M7 files written from the notebook")
else:
    print("M7 already in the repository checkout")
sh(f"{sys.executable} -m pip install -q -e . pytest")
print(platform.python_version(), subprocess.run("git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# Find each file the repo's manifest lists for the two mapped folders (by name, anywhere under /kaggle/input), check its SHA-256, copy it into place.
MAPPED = ("data/raw/hupa_ucm/Preprocessed/", "data/raw/brist1d/device_data/processed_state/")
want = [r for r in csv.DictReader(open("data/manifest.csv")) if r["path"].endswith(".csv") and r["path"].startswith(MAPPED)]
index = {}
for f in INPUT_ROOT.rglob("*.csv"):
    index.setdefault(f.name, []).append(f)
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
missing, bad = [], []
for r in want:
    name = pathlib.Path(r["path"]).name; cands = index.get(name, [])
    pick = next((c for c in cands if sha(c) == r["sha256"]), None) if VERIFY_HASHES else (cands[0] if cands else None)
    if pick is None:
        (bad if cands else missing).append(name); continue
    dst = pathlib.Path(r["path"]); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(pick, dst)
n = {k: sum(1 for r in want if r["path"].startswith(k)) for k in MAPPED}
print("expected files:", n, "| missing:", missing, "| found but hash differs:", bad)
if missing or bad:
    raise SystemExit("attach the unmodified HUPA-UCM (25 Preprocessed CSVs) and BrisT1D-Open (20 processed_state CSVs) datasets")
print("all raw files present" + (" and match the registered SHA-256" if VERIFY_HASHES else " (hash check OFF)"))

In [ ]:
for ds in ("hupa_ucm", "brist1d"):
    sh(f"{sys.executable} -m iris.tools.ingest_dataset {ds} --mode provisional")
sh(f"{sys.executable} -m iris.tools.ingest_dataset brist1d --mode provisional --combine")
sh(f"{sys.executable} -m iris.tools.validate_person_day data/processed/person_day.parquet")

In [ ]:
# BLOCKED is a legitimate outcome (the repo refuses to invent an input); it is printed, never hidden
sh(f"{sys.executable} -m pytest -q tests/leakage/test_next_day_features.py", check=False)
for exp in ("M4_leakage", "M1_baseline", "M7_next_day_tdd_ml", "M6_circadian_recovery"):
    sh(f"{sys.executable} -m iris.tools.run_experiment {exp} --mode provisional", check=False)

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
info = lambda r: json.loads((r / "provenance.json").read_text())["run"]
runs = sorted(pathlib.Path("results/runs").glob("*"), key=lambda p: p.stat().st_mtime)
latest = {}
for r in runs: latest[info(r)["experiment"]] = r
for e, r in latest.items():
    print(f"{e:28s} {info(r)['status']:10s} {info(r)['run_mode']:12s} {r.name}  warnings={info(r)['warnings']}")
t4 = pd.read_parquet(latest["M4_leakage"] / "tables" / "leakage_tests.parquet"); print("\nM4 leakage checks passed:", int(t4.passed.sum()), "/", len(t4))
print("M1 baseline:"); print(pd.read_parquet(latest["M1_baseline"] / "tables" / "baseline_scores.parquet").round(4).to_string(index=False))
m7 = latest["M7_next_day_tdd_ml"]
T = lambda n: pd.read_parquet(m7 / "tables" / f"{n}.parquet")
print("\n== M7 cohort flow");  print(T("cohort_flow").to_string(index=False))
print("\n== M7 feature audit"); print(T("feature_audit").round(3).to_string(index=False))
print("\n== M7 scores (primary = LeaveOneGroupOut; skill > 0 means lower error than the baseline)"); s = T("scores"); print(s[s.scheme == "LeaveOneGroupOut"][["model","n","mae_u","rmse_u","mape_pct","skill_vs_P1_mae","skill_vs_P7_mae","skill_vs_P7_lo","skill_vs_P7_hi"]].round(3).to_string(index=False))
print("\n== M7 VERDICT (fixed rule, primary scheme)"); v = T("verdict"); print(v[v.primary].round(3).to_string(index=False))
print("\n== M7 placebo (must show no skill)"); print(T("placebo_control").round(3).to_string(index=False))
print("\n== M7 by dataset"); d = T("scores_by_dataset"); print(d[["dataset","model","n_people","n","mae_u","mape_pct"]].round(3).to_string(index=False))

In [ ]:
out = WORK_ROOT / "out"; out.mkdir(exist_ok=True)
sh(f"{sys.executable} -m pip freeze > {out}/pip_freeze.txt")
json.dump({"repo": REPO_URL, "base_commit": BASE_COMMIT, "python": platform.python_version(), "platform": platform.platform(),
           "mode": "provisional", "hash_check": VERIFY_HASHES}, open(out / "environment.json", "w"), indent=2)
shutil.copytree("results/runs", out / "runs", dirs_exist_ok=True)
shutil.copytree("data/processed", out / "processed_qc", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.parquet"))   # QC reports only
shutil.make_archive(str(WORK_ROOT / "iris_kaggle_outputs"), "zip", out)
print("download iris_kaggle_outputs.zip from the Output tab (no raw data inside)")